## US Presidential Elections (2000 - 2024): Transform and Load

Sources:
- MIT Election Data and Science Lab. (2018). County Presidential Election Returns 2000-2024 (Version V20) [dataset]. Harvard Dataverse. https://doi.org/10.7910/DVN/VOQCHQ
- U.S. Census Bureau. (2000–2024). TIGER/Line shapefiles [Data set]. U.S. Department of Commerce. https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

### Imports

In [25]:
import os
import pandas as pd

### Read from CSV

In [26]:
DATA_PATH = os.path.join("..", "..", "..", "data")
pres_election_df = pd.read_csv(os.path.join(DATA_PATH, "processed", "us_county_pres_2000-2024_processed.csv"))
state_borders_df = pd.read_csv(os.path.join(DATA_PATH, "processed", "us_state_borders_2000-2024_processed.csv"))
county_borders_df = pd.read_csv(os.path.join(DATA_PATH, "processed", "us_county_borders_2000-2024_processed.csv"))

In [27]:
pres_election_df.head()

,year,state,county_name,county_fips,candidate,party,votes,total_votes
0,2000,ALABAMA,AUTAUGA,1001,AL GORE,DEMOCRATIC PARTY,4942,17208
1,2000,ALABAMA,AUTAUGA,1001,GEORGE W. BUSH,REPUBLICAN PARTY,11993,17208
2,2000,ALABAMA,AUTAUGA,1001,OTHER,OTHER,113,17208
3,2000,ALABAMA,AUTAUGA,1001,RALPH NADER,GREEN PARTY,160,17208
4,2004,ALABAMA,AUTAUGA,1001,GEORGE W. BUSH,REPUBLICAN PARTY,15196,20081


In [28]:
state_borders_df.head()

,state,year,geometry
0,ALABAMA,2000,"POLYGON ((-85.513612 34.523824999999995, -85.5..."
1,ALASKA,2000,MULTIPOLYGON (((177.4459323009414 52.111340904...
2,ARIZONA,2000,"POLYGON ((-113.915986 36.999976, -113.91555 36..."
3,ARKANSAS,2000,"MULTIPOLYGON (((-92.114618 36.498036, -92.1146..."
4,CALIFORNIA,2000,"MULTIPOLYGON (((-119.000932 33.535895, -119.00..."


In [29]:
county_borders_df.head()

,county_fips,year,geometry
0,1001,2000,"POLYGON ((-86.62619 32.706381, -86.62498 32.70..."
1,1001,2008,"POLYGON ((-86.41244999999999 32.57084, -86.412..."
2,1001,2012,"POLYGON ((-86.921196 32.657542, -86.920929 32...."
3,1001,2016,POLYGON ((-86.90309599999999 32.54062599999999...
4,1001,2020,POLYGON ((-86.90309599999999 32.54062599999999...


### Table Creation

In [30]:
countries_df = pd.DataFrame({"name": ["UNITED STATES"]})
countries_df

,name
0,UNITED STATES


In [31]:
divisions_df = pres_election_df[["state"]]
divisions_df = divisions_df \
    .drop_duplicates() \
    .reset_index(drop=True)
divisions_df["country"] = "UNITED STATES"
divisions_df["type"] = "STATE"
divisions_df = divisions_df.rename(columns={"state": "name"})
divisions_df.head()

,name,country,type
0,ALABAMA,UNITED STATES,STATE
1,ALASKA,UNITED STATES,STATE
2,ARIZONA,UNITED STATES,STATE
3,ARKANSAS,UNITED STATES,STATE
4,CALIFORNIA,UNITED STATES,STATE


In [32]:
division_borders_df = divisions_df.merge(state_borders_df, left_on="name", right_on="state", how="left", indicator=True) \
    .drop(columns=["type", "state"]) \
    .rename(columns={"year": "year_drawn"})

# Verify that there are no unmatched rows wrt divisions_df - ie. all states in divisions_df have border data
division_borders_unmatched_rows = division_borders_df[division_borders_df["_merge"] == "left_only"]
division_borders_unmatched_rows

,name,country,year_drawn,geometry,_merge


In [33]:
division_borders_df = division_borders_df.drop(columns=["_merge"])
division_borders_df.head()

,name,country,year_drawn,geometry
0,ALABAMA,UNITED STATES,2000,"POLYGON ((-85.513612 34.523824999999995, -85.5..."
1,ALABAMA,UNITED STATES,2008,"POLYGON ((-86.341764 34.991386, -86.3401279999..."
2,ALABAMA,UNITED STATES,2012,"POLYGON ((-88.406201 30.589707999999998, -88.4..."
3,ALABAMA,UNITED STATES,2016,"POLYGON ((-88.139988 34.581703, -88.139969 34...."
4,ALABAMA,UNITED STATES,2020,"POLYGON ((-88.139988 34.581703, -88.139969 34...."


In [34]:
regions_df = pres_election_df[["state", "county_fips", "county_name"]]
regions_df = regions_df \
    .drop_duplicates() \
    .reset_index(drop=True)
regions_df["country"] = "UNITED STATES"

STATE_TO_REGION_MAPPING = {
    "ALASKA": "STATE HOUSE DISTRICT",
    "LOUISIANA": "PARISH",
    "DISTRICT OF COLUMBIA": "FEDERAL DISTRICT"
}

# There are actually cities (eg. St Louis City, Kansas City MO) which were taken to be counties by the
# MIT dataset. So, we classify all the regions from the MIT dataset as "COUNTY OR EQUIVALENT"
regions_df["type"] = regions_df.apply(lambda row: STATE_TO_REGION_MAPPING.get(row["state"], "COUNTY OR EQUIVALENT"), axis=1)
regions_df = regions_df.drop(columns=["state"])
regions_df = regions_df.rename(columns={"county_fips": "id", "county_name": "name"})
regions_df.head()

,id,name,country,type
0,1001,AUTAUGA,UNITED STATES,COUNTY OR EQUIVALENT
1,1003,BALDWIN,UNITED STATES,COUNTY OR EQUIVALENT
2,1005,BARBOUR,UNITED STATES,COUNTY OR EQUIVALENT
3,1007,BIBB,UNITED STATES,COUNTY OR EQUIVALENT
4,1009,BLOUNT,UNITED STATES,COUNTY OR EQUIVALENT


In [35]:
region_borders_df = regions_df.merge(county_borders_df, left_on="id", right_on="county_fips", how="left", indicator=True) \
    .drop(columns=["name", "country", "type", "county_fips"]) \
    .rename(columns={"year": "year_drawn"})

# Verify that there are no unmatched rows wrt regions_df - ie. all regions in regions_df have border data
region_borders_unmatched_rows = region_borders_df[region_borders_df["_merge"] == "left_only"]
region_borders_unmatched_rows

,id,year_drawn,geometry,_merge


In [36]:
region_borders_df = region_borders_df.drop(columns=["_merge"])
region_borders_df.head()

,id,year_drawn,geometry
0,1001,2000,"POLYGON ((-86.62619 32.706381, -86.62498 32.70..."
1,1001,2008,"POLYGON ((-86.41244999999999 32.57084, -86.412..."
2,1001,2012,"POLYGON ((-86.921196 32.657542, -86.920929 32...."
3,1001,2016,POLYGON ((-86.90309599999999 32.54062599999999...
4,1001,2020,POLYGON ((-86.90309599999999 32.54062599999999...


In [37]:
OTHER_PARTY_NAME = "OTHER"
EXCEPTIONAL_PARTY_NAME = "EXCEPTIONAL"

candidates_df = pres_election_df[
    ~pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]
candidates_df = candidates_df[["candidate"]] \
    .drop_duplicates() \
    .reset_index(drop=True)
candidates_df = candidates_df.rename(columns={"candidate": "name"})
candidates_df

,name
0,AL GORE
1,GEORGE W. BUSH
2,RALPH NADER
3,JOHN KERRY
4,BARACK OBAMA
5,JOHN MCCAIN
6,MITT ROMNEY
7,DONALD TRUMP
8,HILLARY CLINTON
9,DONALD J TRUMP


In [38]:
parties_df = pres_election_df[["party"]]
parties_df = parties_df[~parties_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])] \
    .drop_duplicates() \
    .reset_index(drop=True)
parties_df = parties_df.rename(columns={"party": "name"})
parties_df

,name
0,DEMOCRATIC PARTY
1,REPUBLICAN PARTY
2,GREEN PARTY
3,LIBERTARIAN PARTY


In [39]:
elections_df = pres_election_df[["year"]].drop_duplicates().reset_index(drop=True)
elections_df["name"] = elections_df["year"].apply(lambda year: f"{year} UNITED STATES PRESIDENTIAL ELECTION")
elections_df["type"] = "PRESIDENTIAL"
elections_df = elections_df[["name", "year", "type"]]
elections_df

,name,year,type
0,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,PRESIDENTIAL
1,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,PRESIDENTIAL
2,2008 UNITED STATES PRESIDENTIAL ELECTION,2008,PRESIDENTIAL
3,2012 UNITED STATES PRESIDENTIAL ELECTION,2012,PRESIDENTIAL
4,2016 UNITED STATES PRESIDENTIAL ELECTION,2016,PRESIDENTIAL
5,2020 UNITED STATES PRESIDENTIAL ELECTION,2020,PRESIDENTIAL
6,2024 UNITED STATES PRESIDENTIAL ELECTION,2024,PRESIDENTIAL


In [40]:
races_df = pres_election_df[["year", "state"]].drop_duplicates() \
    .reset_index(drop=True) \
    .merge(elections_df, on=["year"], how="inner") \
    .drop(columns=["type"]) \
    .rename(columns={"year": "election_year", "state": "division_name", "name": "election_name"})

races_df = races_df[["election_name", "election_year", "division_name"]]
races_df.head()

,election_name,election_year,division_name
0,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA
1,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA
2,2008 UNITED STATES PRESIDENTIAL ELECTION,2008,ALABAMA
3,2012 UNITED STATES PRESIDENTIAL ELECTION,2012,ALABAMA
4,2016 UNITED STATES PRESIDENTIAL ELECTION,2016,ALABAMA


In [41]:
participations_df = pres_election_df[
    ~pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]

participations_df = participations_df[["year", "state", "candidate", "votes"]] \
    .groupby(["candidate", "year", "state"])["votes"] \
    .sum() \
    .reset_index()

participations_df = participations_df.rename(columns={
    "year": "election_year", "state": "division_name", "votes": "total_votes"
}).merge(races_df, on=["election_year", "division_name"], how="inner")

participations_df = participations_df[[
    "candidate",
    "election_name",
    "election_year",
    "division_name",
    "total_votes"
]]

participations_df

,candidate,election_name,election_year,division_name,total_votes
0,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,695602
1,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALASKA,79004
2,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ARIZONA,685341
3,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ARKANSAS,422768
4,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,CALIFORNIA,5861203
...,...,...,...,...,...
828,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,VIRGINIA,59373
829,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,WASHINGTON,103002
830,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,WEST VIRGINIA,10680
831,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,WISCONSIN,94070


In [42]:
participation_parties_df = pres_election_df[
    ~pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]

participation_parties_df = participation_parties_df[["candidate", "party", "year", "state"]] \
    .drop_duplicates() \
    .reset_index(drop=True)

participation_parties_df = participation_parties_df.merge(elections_df, on=["year"], how="inner") \
    .drop(columns=["type"])

participation_parties_df = participation_parties_df.rename(columns={
    "year": "election_year", "state": "division_name", "name": "election_name"
})

participation_parties_df = participation_parties_df[[
    "candidate",
    "election_name",
    "election_year",
    "division_name",
    "party"
]]

participation_parties_df.head()

,candidate,election_name,election_year,division_name,party
0,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,DEMOCRATIC PARTY
1,GEORGE W. BUSH,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,REPUBLICAN PARTY
2,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,GREEN PARTY
3,GEORGE W. BUSH,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA,REPUBLICAN PARTY
4,JOHN KERRY,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA,DEMOCRATIC PARTY


In [43]:
other_vote_types_df = pd.DataFrame({"type": ["OTHER CANDIDATE", "SPOILED", "UNDERVOTES", "OVERVOTES"]})
other_vote_types_df

,type
0,OTHER CANDIDATE
1,SPOILED
2,UNDERVOTES
3,OVERVOTES


In [44]:
other_racewide_results_df = pres_election_df[
    pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]

other_racewide_results_df = other_racewide_results_df \
    .groupby(["year", "state", "candidate"])["votes"] \
    .sum() \
    .reset_index() \
    .merge(elections_df, on=["year"], how="inner") \
    .drop(columns=["type"]) \
    .rename(columns={
        "year": "election_year", "state": "division_name", "candidate": "type", "name": "election_name"
    })

other_racewide_results_df = other_racewide_results_df[[
    "election_name",
    "election_year",
    "division_name",
    "type",
    "votes"
]]

other_racewide_results_df.head()

,election_name,election_year,division_name,type,votes
0,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,OTHER,14191
1,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALASKA,OTHER,10381
2,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ARIZONA,OTHER,21475
3,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ARKANSAS,OTHER,12652
4,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,CALIFORNIA,OTHER,118517


In [45]:
participation_regional_results_df = pres_election_df[
    ~pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]

participation_regional_results_df = participation_regional_results_df.merge(elections_df, on=["year"], how="inner") \
    .drop(columns=["party", "county_name", "total_votes", "type"])

participation_regional_results_df = participation_regional_results_df.rename(columns={
    "state": "division_name", "year": "election_year", "county_fips": "region_id", "name": "election_name"
})

participation_regional_results_df = participation_regional_results_df[[
    "candidate",
    "election_name",
    "election_year",
    "division_name",
    "region_id",
    "votes"
]]

participation_regional_results_df.head()

,candidate,election_name,election_year,division_name,region_id,votes
0,AL GORE,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,1001,4942
1,GEORGE W. BUSH,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,1001,11993
2,RALPH NADER,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,1001,160
3,GEORGE W. BUSH,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA,1001,15196
4,JOHN KERRY,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA,1001,4758


In [46]:
other_regional_results_df = pres_election_df[
    pres_election_df["party"].isin([OTHER_PARTY_NAME, EXCEPTIONAL_PARTY_NAME])
]

other_regional_results_df = other_regional_results_df.merge(elections_df, on=["year"], how="inner") \
    .drop(columns=["party", "county_name", "total_votes", "type"])

other_regional_results_df = other_regional_results_df.rename(columns={
    "year": "election_year", "state": "division_name", "candidate": "type", "county_fips": "region_id", "name": "election_name"
})

other_regional_results_df = other_regional_results_df[[
    "election_name",
    "election_year",
    "division_name",
    "region_id",
    "type",
    "votes"
]]

other_regional_results_df.head()

,election_name,election_year,division_name,region_id,type,votes
0,2000 UNITED STATES PRESIDENTIAL ELECTION,2000,ALABAMA,1001,OTHER,113
1,2004 UNITED STATES PRESIDENTIAL ELECTION,2004,ALABAMA,1001,OTHER,127
2,2008 UNITED STATES PRESIDENTIAL ELECTION,2008,ALABAMA,1001,OTHER,145
3,2012 UNITED STATES PRESIDENTIAL ELECTION,2012,ALABAMA,1001,OTHER,190
4,2016 UNITED STATES PRESIDENTIAL ELECTION,2016,ALABAMA,1001,OTHER,865


### Verify Total Votes

In [47]:
# Verify that regional results of each candidacy sum up to its total vote tally within the given division
participation_regional_totals = participation_regional_results_df \
    .groupby(
        ["election_name", "election_year", "division_name", "candidate"],
        dropna=False,
        as_index=False
    )["votes"] \
    .sum() \
    .rename(columns={"votes": "regional_votes"})

candidacy_comparison = participations_df.merge(
    participation_regional_totals,
    on=["election_name", "election_year", "division_name", "candidate"],
    how="outer",
    indicator=True
)

candidacy_comparison["difference"] = candidacy_comparison["total_votes"] - candidacy_comparison["regional_votes"]

candicacy_mismatches = candidacy_comparison[
    candidacy_comparison["difference"] != 0
]

candicacy_mismatches

,candidate,election_name,election_year,division_name,total_votes,regional_votes,_merge,difference


In [48]:
# Verify that other regional results sum up to its total vote tally within the given division
other_regional_totals = other_regional_results_df \
    .groupby(
        ["election_name", "election_year", "division_name", "type"],
        dropna=False,
        as_index=False
    )["votes"] \
    .sum() \
    .rename(columns={"votes": "regional_votes"})

other_comparison = other_racewide_results_df.merge(
    other_regional_totals,
    on=["election_name", "election_year", "division_name", "type"],
    how="outer",
    indicator=True
)

other_comparison["difference"] = other_comparison["votes"] - other_comparison["regional_votes"]

other_mismatches = other_comparison[
    other_comparison["difference"] != 0
]

other_mismatches

,election_name,election_year,division_name,type,votes,regional_votes,_merge,difference


### Insert Into Database